In [33]:
from pypdf import PdfReader
import re
from pprint import pprint
from sentence_transformers import SentenceTransformer
import os 
from dotenv import load_dotenv
from google import genai
from google.genai import types
from ollama import chat
import time

In [34]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [35]:
def clean_text(full_text):
    #removes boilerplate 
    full_text = re.sub(r"This content.*?/terms", "", full_text, flags=re.DOTALL)
    full_text = re.sub(r"THE MATHE.*?ERICA", "", full_text, flags=re.DOTALL)
    full_text = re.sub(r"JUGGLING.*?2005", "", full_text, flags=re.DOTALL)
    text_list = []
    for line in full_text.split("\n"):
        if "\x00" in line: #removes nullspace
            line = line.replace("\x00",'')
            text_list.append(line)
        elif not line.strip():  # empty or only whitespace
            continue
        else: #adds items back to list
            text_list.append(line)
    cleaned_text = "\n".join(text_list)
    return cleaned_text

In [36]:
def pdf_to_text(file_name):
    '''Takes in file name and processes it as a text string'''
    full_text = ''
    with open(file_name,"rb") as file:
        reader = PdfReader(file)
        # Loop through all pages
        for page in reader.pages: #page is an object of pageobject class
            page_text = page.extract_text()
            full_text+=" \n" + page_text
    return full_text


In [37]:
file_name = "Warrington-JugglingProbabilities-2005.pdf"
text = pdf_to_text(file_name)
cleaned_text = clean_text(text)

In [38]:

def chunk_text(text, chunk_size, overlap):
    '''chunks text and returns a list of chunk texts ['chunk1','chunk2',...]'''
    text_list = []
    i = 0
    while i < len(text):
        # print(f'Start {i}, End {i + chunk_size}')
        text_list.append( text[i:i+chunk_size])
        i = i + (chunk_size - overlap)
    return text_list
        

In [39]:
chunks = chunk_text(cleaned_text, 800, 100)
chunks
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5455.20it/s]


In [40]:
import chromadb
import datetime
from sentence_transformers import SentenceTransformer

#Loading a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
def embed_chunks(chunks,model):
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_or_create_collection(name="juggling_paper", 
                                                        metadata={
                                                                    "description": "chroma collection for pdf chunks",
                                                                    "created": str(datetime.datetime.now())
                                                                }
                                                        )
    if collection.count() == 0: #optimizing the function to prevent unnecessary encoding on reruns
        collection.add(
            ids = [f"id{i}" for i in range(len(chunks))],
            documents = chunks, 
            embeddings = model.encode(chunks) # Calculate embeddings by calling model.encode()
        )
        
    return collection
collection = embed_chunks(chunk_text(cleaned_text, 800, 100),model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11592.38it/s]


In [41]:
import numpy as np
def search(question, collection, model, k=5):
    '''returns top k answers to question based on similarity score'''
    q_embedding = model.encode(question)
    result = collection.query(query_embeddings = q_embedding,
                        n_results=k)
    text_list = result['documents'][0]
    score_list = result['distances'][0]
    return [{'text': t, 'score': s} for t, s in zip(text_list, score_list)]


search('what is a juggling process?', collection, model)

[{'text': 'bility of making each legal throw. \n There are countless ways to make these specifications, but we begin our investigations \n with the model that most closely mirrors what people envision as juggling. In the \n next section, we describe the juggling universe determined by this standard model and \n answer our random question. In section 3 we generalize the notion of a state in order to \n give a simple justification for our answer. Finally, in section 4 we describe variations \n of this model. \n 2. ] \n 105\n We, however, intend to strip juggling down to what is (arguably) its essentials. Our\n juggler will juggle only balls and only in the air. He will not attempt jokes. Whether\n a hand is under a leg or behind the back when making a catch will not be noted. We\n record only the order in which th',
  'score': 0.7659505605697632},
 {'text': 'ows to heights that would not lead to \n two balls landing at the same time. In the second model, however, we allow for this \n eve

In [43]:
#testing the results
for q in [
    "What is a Markov chain?",
    "Who is the author?",
    "How does juggling work?",
    "Banana sandwich recipe",
]:
    print(f"\n=== {q} ===")
    results = search(q, collection, model, k=3)
    for r in results:
        print(f"  ----{r['score']:.3f}--- | {r['text'][:100]}")


=== What is a Markov chain? ===
  ----0.824--- | When MC(Sth,f, P) is in state i, we define our new process to be in the state 
 [rh (~)]. It follows
  ----0.869--- | obabilities depend only on the current state. To de- 
 scribe a Markov chain, we need to know the po
  ----1.015--- | atter Markov chain. The 
 first step is to find the vector /3 of steady-state probabilities for MC(S

=== Who is the author? ===
  ----1.551--- | Juggling Probabilities 
Author(s): Gregory S. Warrington 
Source: The American Mathematical Monthly 
  ----1.664--- |  He is reputed to be the first person to 
 juggle five clubs. Reliable details of his life are outcl
  ----1.692--- | 7
 Figure 2. The state graph G5,2.
 Let us revisit the scenario introduced at the beginning of the p

=== How does juggling work? ===
  ----0.683--- | bility of making each legal throw. 
 There are countless ways to make these specifications, but we b
  ----0.847--- |  leg or behind the back when making a catch will not be noted. 

In [44]:
def build_prompt(question, retrieved_chunks):
    '''Pulls out text from retrieved chunks and labels final string as question and context'''
    text_list = [item['text'] for item in retrieved_chunks]
    return 'Context: ' + '---\n---'.join(text_list) + '---\n---' 'Question: ' + question

In [45]:
results = search(q, cole, model, k=8)
q = "What is a Markov chain?"
c = build_prompt(q, results)

NameError: name 'cole' is not defined

In [46]:
import time

def ask_llm_gemini(question, retrieved_chunks, client, max_retries=3):
    prompt = build_prompt(question, retrieved_chunks)
    
    for attempt in range(max_retries):
        time.sleep(20)
        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                config=types.GenerateContentConfig(
                    system_instruction="You are a helpful assistant. Use only the provided context to answer the user's question. If the context does not contain the answer, say so honestly."
                ),
                contents=prompt,
            )
            return response.text
        except Exception as e:
            if attempt == max_retries - 1:
                raise  # last attempt — let it crash
            wait_time = 5 ** attempt  # 1s, 25s, 125s — exponential backoff a^x
            print(f"  API error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

In [47]:
import time
import ollama
def ask_llm_deep_seek(question, retrieved_chunks,model='deepseek-r1'):
    prompt = build_prompt(question, retrieved_chunks)
    
    response = chat(
            model=model,
            messages=[{'role': 'system', 'content': 'You are a helpful assistant. Use only the provided context'},
                {'role': 'user', 'content': prompt}],
        )
    return response.message.content
    

In [48]:
test_queries = [
    {
        "question": "Who is the author of this paper?",
        "category": "easy_factual",
        "expected_keywords": ["Warrington","Gregory"],   # answer must include at least one of these
        "should_answer": True,                  # the doc contains this info
    },
    {
        "question": "What year was the paper published?",
        "category": "easy_factual",
        "expected_keywords": ["2005", "February"],   # answer must include at least one of these
        "should_answer": True,                  # the doc contains this info
    },
    {
        "question": "What are the references",
        "category": "metadata",
        "expected_keywords": ["Buhler","Kemeny","jugglingdb","Eisenbud","Graham",
        "Ehrenborg","Kamstra"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "How many pages are there?",
        "category": "metadata",
        "expected_keywords": ["14","15"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is a markov chain",
        "category": "definition",
        "expected_keywords": ['discrete time',"random","transition", "probabilities"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is the juggling process",
        "category": "definition",
        "expected_keywords": ["throw","ball","hand"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is theorem 1, explain it?",
        "category": "detail",
        "expected_keywords": ["juggling","balls","fraction", "V","S_t_{h,f}","binom{h+1}{f+1}","b balls","writing f", "h-b","h+1/f+1"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What are the differences between the standard model and the add-drop model?",
        "category": "synthesis",
        "expected_keywords": ["generalizes standard juggling","assistant helping", "removing restriction on v_1","height 0","add","insert"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What's the main idea?",
        "category": "tricky_semantic",
        "expected_keywords": ["random","fraction","time"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Who first considered the juggling state graph?",
        "category": "specific_question",
        "expected_keywords": ["Boyce"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Banana sandwich recipe",
        "category": "negative",
        "expected_keywords": [],
        "should_answer": False,                 # model should refuse
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Did the egg come before the chicken?",
        "category": "negative",
        "expected_keywords": [],
        "should_answer": False,                 # model should refuse
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    
]

In [49]:
def score_query(test_query, answer):
    answer_lower = answer.lower()
    
    if test_query["should_answer"]:
        # Positive test: look for any expected keyword in the answer
        matches = [kw for kw in test_query["expected_keywords"] 
                   if kw.lower() in answer_lower]
        passed = len(matches) > 0
        if passed:
            reason = f"found keywords: {matches}"
        else:
            reason = f"missing all expected keywords: {test_query['expected_keywords']}"
    else:
        # Negative test: look for any refusal phrase in the answer
        refusals = [kw for kw in test_query["refusal_keywords"] 
                    if kw.lower() in answer_lower]
        passed = len(refusals) > 0
        if passed:
            reason = f"refused with: {refusals}"
        else:
            reason = "did not refuse"
    
    return {
        "question": test_query["question"],
        "category": test_query["category"],
        "answer": answer,
        "passed": passed,
        "reason": reason,
    }

In [50]:
import json
#storing test_queries as a text file
with open('test_queries.txt', 'w') as f:
    json.dump(test_queries, f, indent=4) # indent=4 makes it human-readable


In [51]:
# 1. Positive test, model answered correctly
print(score_query(test_queries[0], "The author is Gregory S. Warrington."))

# 2. Positive test, model answered wrong
print(score_query(test_queries[0], "I don't know."))

# 3. Negative test, model refused
neg_test = next(t for t in test_queries if not t["should_answer"])
print(score_query(neg_test, "The document does not contain information about this."))

# 4. Negative test, model failed to refuse
print(score_query(neg_test, "Here is a banana sandwich recipe."))

{'question': 'Who is the author of this paper?', 'category': 'easy_factual', 'answer': 'The author is Gregory S. Warrington.', 'passed': True, 'reason': "found keywords: ['Warrington', 'Gregory']"}
{'question': 'Who is the author of this paper?', 'category': 'easy_factual', 'answer': "I don't know.", 'passed': False, 'reason': "missing all expected keywords: ['Warrington', 'Gregory']"}
{'question': 'Banana sandwich recipe', 'category': 'negative', 'answer': 'The document does not contain information about this.', 'passed': True, 'reason': "refused with: ['does not']"}
{'question': 'Banana sandwich recipe', 'category': 'negative', 'answer': 'Here is a banana sandwich recipe.', 'passed': False, 'reason': 'did not refuse'}


In [ ]:
def run_eval(test_queries, collection, model, client, agent='gemini'):
    """Run all queries through the pipeline. Return list of result dicts."""
    # For each query: call search → ask_llm → score_query → collect
    eval_results = []
    for query in test_queries:
    
        question = query['question']
        results = search(question, collection, model, k=3)
        start_time = time.perf_counter()
        
        if agent == 'gemini':
            answer = ask_llm_gemini(question, results,client)
        else:
            answer = ask_llm_deep_seek(question, results)
        end_time = time.perf_counter()
        
        if agent == 'gemini':
            elapsed_time = end_time - start_time - 20 #account for sleep time in gemini call
        else:
            elapsed_time = end_time - start_time #time taken to 
        print(f"LLM response time: {elapsed_time:.4f} seconds")

        score_result = score_query(query,answer)
        eval_results.append(score_result)
        print(f"[{len(eval_results)}/{len(test_queries)}] {'Correct' if score_result['passed'] else 'Incorrect'} {question}")
    return eval_results
    

In [56]:
client = genai.Client()
eval_results_gemini = run_eval(test_queries, collection, model, client)

LLM response time: 2.8199 seconds
[1/12] Correct Who is the author of this paper?
LLM response time: 2.7414 seconds
[2/12] Correct What year was the paper published?
LLM response time: 4.3415 seconds
[3/12] Incorrect What are the references
LLM response time: 4.5628 seconds
[4/12] Correct How many pages are there?
LLM response time: 5.1096 seconds
[5/12] Correct What is a markov chain
LLM response time: 5.5404 seconds
[6/12] Correct What is the juggling process
LLM response time: 8.0746 seconds
[7/12] Correct What is theorem 1, explain it?
LLM response time: 8.2230 seconds
[8/12] Correct What are the differences between the standard model and the add-drop model?
LLM response time: 4.5332 seconds
[9/12] Incorrect What's the main idea?
LLM response time: 4.0263 seconds
[10/12] Correct Who first considered the juggling state graph?
LLM response time: 4.5807 seconds
[11/12] Correct Banana sandwich recipe
LLM response time: 2.5720 seconds
[12/12] Correct Did the egg come before the chicken?

In [54]:
def print_summary(results):
    """Pretty-print the results: per-query + overall + per-category."""
    for result in results:
        pprint(f"Question: {result['question']}")
        pprint(f"Category: {result['category']}")
        pprint(f"Passed: {result['passed']}")
        pprint(f"Answer: {result['answer']}")

In [57]:
eval_results_deepseek = run_eval(test_queries, collection, model, client, agent = 'deep_seek')

LLM response time: 19.3672 seconds
[1/12] Correct Who is the author of this paper?
LLM response time: 15.5362 seconds
[2/12] Correct What year was the paper published?
LLM response time: 27.2208 seconds
[3/12] Incorrect What are the references
LLM response time: 19.4202 seconds
[4/12] Correct How many pages are there?
LLM response time: 26.7274 seconds
[5/12] Correct What is a markov chain
LLM response time: 50.9722 seconds
[6/12] Correct What is the juggling process
LLM response time: 29.1186 seconds
[7/12] Correct What is theorem 1, explain it?
LLM response time: 58.9689 seconds
[8/12] Correct What are the differences between the standard model and the add-drop model?
LLM response time: 30.1726 seconds
[9/12] Incorrect What's the main idea?
LLM response time: 15.7687 seconds
[10/12] Correct Who first considered the juggling state graph?
LLM response time: 23.3012 seconds
[11/12] Incorrect Banana sandwich recipe
LLM response time: 59.9170 seconds
[12/12] Correct Did the egg come befor

In [ ]:
print_summary(eval_results_deepseek)

### Takeaways:
- Local LLMs work end-to-end on laptop
- Quality drops are category-specific — easy questions stay easy, hard questions get harder
- Reasoning models like deepseek-r1 trade tons of latency for thinking and aren't always the right tool for RAG. Or possibly require more parameters to perform better. 
- Evaluation is not entirely correct. Deepseek said it doesnt come from context but answered anyway but this was not picked up by the test as it doesnt check properly.
- Gemini flash performed better than deepseek r1 8b.

In [58]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
async def root():
    return {"message": "Hello World"}